In [1]:
from google import genai
from google.genai import types
from datasets import load_dataset
import requests
import time
import re

In [2]:
!pip install datasets -q

In [3]:
from datasets import load_dataset

In [ ]:
client = genai.Client(api_key="###############") # change with own key

In [ ]:
#PLAN: Do 2 rounds of debate and cut it off. 

#OR should be just do a single round?

#We'll store each round and not do everything at once. 

#It should go: pos -> neg -> mod -> pos -> neg -> mod -> extract answer




#API KEYS 



def positive_debate(row, idx):
    # Download the image
    try:
        image_response = requests.get(row["image_url"])
    except Exception as e:
        print(f"Error downloading image at index {idx}: {e}")
        row['positive'] = None
        return row

    if image_response.status_code != 200:
        print(f"Failed to download image at index {idx}. Status code: {image_response.status_code}")
        row['positive'] = None
        return row

    image_bytes = image_response.content

    pos_evidence_graph = row["KG"]
    question = row["question"] + " Choices: " + row["choices"]
    
    prompt_text = f"""As Ben, a high school student with an impressive academic record, is respected by peers for his knowledge and logical thinking., you are assigned as an affirmative debater and have been provided with an evidence graph {pos_evidence_graph}  
    for answering the question {question} related to the image. Try to enhance the graph by incorporating your insights  
    towards an optimal solution. Please ensure adherence to the following constraints: Size: The graph must not be empty. Please restrict the maximum number of objects in the graph to 20., Relevence: The objects and relations within the graph should be pertinent to addressing the question., 
    Compactness: The graph must be compact and in the JSON format. Scene graph: """

    max_retries = 3
    attempt = 0
    response = None

    while attempt < max_retries:
        try:
            response = client.models.generate_content(
                contents=[
                    types.Part.from_text(prompt_text),
                    types.Part.from_bytes(image_bytes, mime_type="image/png")
                ],
                model="gemini-2.0-flash"
            )
            # Successfully obtained response, exit retry loop
            break
        except Exception as e:
            error_message = str(e)
            # Check if error is a timeout or resource exhaustion error (429)
            if "408" in error_message or "RESOURCE_EXHAUSTED" in error_message or "429" in error_message:
                print(f"Index {idx}: {error_message} on attempt {attempt+1}. Retrying in 10 seconds...")
                time.sleep(15)
                attempt += 1
            else:
                print(f"Index {idx}: Unexpected error: {error_message}. Skipping this row.")
                attempt = max_retries  # exit loop; will mark as None
    # If response is still None after retries, mark KG as None
    if response is None:
        print(f"Index {idx}: Max retries reached or error occurred. Setting KG to None.")
        row['positive'] = None
    else:
        try:
            scene_graph = response.candidates[0].content.parts
            row['positive'] = scene_graph[0].text
        except Exception as e:
            print(f"Index {idx}: Error processing response: {e}. Setting KG to None.")
            row['positive'] = None

    time.sleep(5)
    return row
    
def negative_debate(row, idx):
    # Download the image
    try:
        image_response = requests.get(row["image_url"])
    except Exception as e:
        print(f"Error downloading image at index {idx}: {e}")
        row['negative'] = None
        return row

    if image_response.status_code != 200:
        print(f"Failed to download image at index {idx}. Status code: {image_response.status_code}")
        row['negative'] = None
        return row

    image_bytes = image_response.content

    pos_evidence_graph = row["positive"]
    question = row["question"] + " Choices: " + row["choices"]
    
    prompt_text = f"""As Ben, a high school student with an impressive academic record, respected by peers for his knowledge and logical thinking, you are assigned as a negative debater and have been provided with 
    an affirmative evidence graph {pos_evidence_graph} for answering the question {question} regarding the image. Try to detect potential flaws and drawbacks of the graph and update it with your insights. 
    Please ensure adherence to the following constraints: Size: The graph must not be empty. Please restrict the maximum number of objects in the graph to 20. 
    Relevance: The objects and relations within the graph should be pertinent to addressing the question. Compactness: The graph must be compact and in JSON format. Scene graph: """

    max_retries = 3
    attempt = 0
    response = None

    while attempt < max_retries:
        try:
            response = client.models.generate_content(
                contents=[
                    types.Part.from_text(prompt_text),
                    types.Part.from_bytes(image_bytes, mime_type="image/png")
                ],
                model="gemini-2.0-flash"
            )
            # Successfully obtained response, exit retry loop
            break
        except Exception as e:
            error_message = str(e)
            # Check if error is a timeout or resource exhaustion error (429)
            if "408" in error_message or "RESOURCE_EXHAUSTED" in error_message or "429" in error_message:
                print(f"Index {idx}: {error_message} on attempt {attempt+1}. Retrying in 10 seconds...")
                time.sleep(15)
                attempt += 1
            else:
                print(f"Index {idx}: Unexpected error: {error_message}. Skipping this row.")
                attempt = max_retries  # exit loop; will mark as None
    # If response is still None after retries, mark KG as None
    if response is None:
        print(f"Index {idx}: Max retries reached or error occurred. Setting KG to None.")
        row['negative'] = None
    else:
        try:
            scene_graph = response.candidates[0].content.parts
            row['negative'] = scene_graph[0].text
        except Exception as e:
            print(f"Index {idx}: Error processing response: {e}. Setting KG to None.")
            row['negative'] = None

    time.sleep(5)
    return row

def mod(row, idx):

    try:
        image_response = requests.get(row["image_url"])
    except Exception as e:
        print(f"Error downloading image at index {idx}: {e}")
        row['mod'] = None
        return row

    if image_response.status_code != 200:
        print(f"Failed to download image at index {idx}. Status code: {image_response.status_code}")
        row['mod'] = None
        return row

    image_bytes = image_response.content

    pos_evidence_graph = row["positive"]
    neg_evidence_graph = row["negative"]
    question = row["question"] + " Choices: " + row["choices"]
    
    prompt_text = f"""As Ben, a high school student with an impressive academic record, respected by peers for his knowledge and logical thinking, you are assigned as a moderator in a  
    debate and have been provided with an affirmative evidence  
    graph {pos_evidence_graph} and a negative evidence graph {neg_evidence_graph} to address the  
    question {question} regarding the image. Try to consolidate  
    the two graphs into a single graph towards the optimal solution,  
    and provide a conclusive answer to the question. Choose only one choice as an answer. Format your answer as /boxed[choice index]/. Scene graph: """

    max_retries = 3
    attempt = 0
    response = None

    while attempt < max_retries:
        try:
            response = client.models.generate_content(
                contents=[
                    types.Part.from_text(prompt_text),
                    types.Part.from_bytes(image_bytes, mime_type="image/png")
                ],
                model="gemini-2.0-flash"
            )
            # Successfully obtained response, exit retry loop
            break
        except Exception as e:
            error_message = str(e)
            # Check if error is a timeout or resource exhaustion error (429)
            if "408" in error_message or "RESOURCE_EXHAUSTED" in error_message or "429" in error_message:
                print(f"Index {idx}: {error_message} on attempt {attempt+1}. Retrying in 10 seconds...")
                time.sleep(15)
                attempt += 1
            else:
                print(f"Index {idx}: Unexpected error: {error_message}. Skipping this row.")
                attempt = max_retries  # exit loop; will mark as None
    # If response is still None after retries, mark KG as None
    if response is None:
        print(f"Index {idx}: Max retries reached or error occurred. Setting KG to None.")
        row['mod'] = None
    else:
        try:
            scene_graph = response.candidates[0].content.parts
            row['mod'] = scene_graph[0].text
        except Exception as e:
            print(f"Index {idx}: Error processing response: {e}. Setting KG to None.")
            row['mod'] = None

    time.sleep(5)
    return row

In [6]:
def extract_answer(row, index):
    #regular expression
    row["mod"] = str(row["mod"])
    match = re.search(r'(/boxed\[|/boxed<)(.*?)(>|\]/)', row["mod"])
    try:
        row["gemini_baseline"] = str(match.group(2))
    except:
        try:
            response = row["mod"]
            sliced = response[response.find("/") + 1:response.rfind("/")] #look within /, which I said it would have to exclose the answer
            if("0" in sliced):
                sliced = "0"
            elif("1" in sliced):
                sliced = "1"
            elif("2" in sliced):
                sliced = "2"
            elif("2" in sliced):
                sliced = "2"
            row["gemini_baseline"] = sliced
        except:
            row["gemini_baseline"] = None
    return row

In [7]:
ScienceQA = load_dataset("/kaggle/input/mod-data")

Generating validation split: 0 examples [00:00, ? examples/s]

In [8]:
validation = ScienceQA["validation"]

In [9]:
# small_dataset = validation.select([0, 10])
# small_dataset
mod_responces = validation.map(extract_answer, with_indices=True)
mod_responces.to_csv("gemini_validation_baselines.csv")

Map:   0%|          | 0/2005 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

20575875

In [10]:
# print(small_dataset['question'][0], "\n--",small_dataset['KG'][0], "\n--",small_dataset['answer'][0], '\n--',small_dataset['choices'][0] )

In [11]:
# response = small_dataset.map(positive_debate, with_indices=True)
# print(response['question'][0], "\n--",response['positive'][0], end='\n***\n'  )


# response = response.map(negative_debater, with_indices=True)
# print(response['question'][0], "\n--",response['negative'][0], "\n--",response['answer'][0], '\n--',response['choices'][0] , end='\n***\n' )


# response = response.map(mod, with_indices=True)
# print(response['question'][0], "\n--",response['mod'][0], end='\n***\n' )




In [12]:
#val_split1 = validation.select(range(1000))
#val_split2 = validation.select(range(1000, len(validation)))

In [13]:
#mod_responces_val_1 = val_split1.map(mod, with_indices=True)

In [14]:
#mod_responces_val_1.to_csv('mod_responces_val_1.csv') 

In [15]:
#time.sleep(120)

In [ ]:
#client = genai.Client(api_key="") # change with own key

In [17]:
#mod_responces_val_2.to_csv('mod_responces_val_2.csv') 